In [1]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
import sympy as smp
from scipy.integrate import quad

In [ ]:


# Plot style: white background with a clearly visible grid (cosmetic only)
sns.set_style("whitegrid")
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.edgecolor"] = "black"
plt.rcParams["grid.color"] = "black"
plt.rcParams["grid.linestyle"] = "-"
plt.rcParams["grid.linewidth"] = 1
plt.rcParams["xtick.color"] = "black"
plt.rcParams["ytick.color"] = "black"
plt.rcParams["axes.labelcolor"] = "black"
plt.rcParams["axes.titlecolor"] = "black"



# Newton-Raphson for Nonlinear Systems of Equations

Three nonlinear systems of increasing dimension (2x2, 2x2, 3x3), each solved with the multivariate
Newton-Raphson method using a numerically approximated Jacobian.

### A. Where do these two circles intersect?
$$
(x - 4)^2 + (y - 4)^2 = 4
$$

$$
x^2 + y^2 = 16
$$

The two expressions were rearranged into the following system, each equation set equal to zero:
$$
(x - 4)^2 + (y - 4)^2 - 4 = 0
$$

$$
x^2 + y^2 - 16 = 0
$$

In [ ]:
import numpy as np

# Functions u(x, y) and v(x, y)
def u(x, y):
    return (x-4)**2 + (y-4)**2 - 4

def v(x, y):
    return x**2 + y**2 - 16

# General parameters
delta = 1e-4
error_tol = 1e-6
max_attempts = 2
max_iterations = 1000

# Approximate derivatives
def du_dx(x, y):
    return (u(x + delta, y) - u(x, y)) / delta

def du_dy(x, y):
    return (u(x, y + delta) - u(x, y)) / delta

def dv_dx(x, y):
    return (v(x + delta, y) - v(x, y)) / delta

def dv_dy(x, y):
    return (v(x, y + delta) - v(x, y)) / delta

# Jacobian determinant
def jacobian(x, y):
    return du_dx(x, y) * dv_dy(x, y) - du_dy(x, y) * dv_dx(x, y)

# Flags and errors
valid_bracket = 0
error_x = 1
error_y = 1
converged = 0
attempts = 0

# First loop: propose initial values
while valid_bracket == 0 and converged == 0 and attempts < max_attempts:
    try:
        x0 = float(input("Give me the initial value for x: "))
        y0 = float(input("Give me the initial value for y: "))
        
        if jacobian(x0, y0) != 0:
            valid_bracket = 1
        
        if u(x0, y0) == 0 and v(x0, y0) == 0:
            converged = 1

        attempts += 1

    except ValueError:
        print("Invalid input. Please enter numeric values.")

# Second loop: iterative method
x_prev, y_prev = x0, y0
iterations = 0

while valid_bracket == 1 and converged == 0 and iterations < max_iterations:
    coef_a = -u(x_prev, y_prev) + du_dx(x_prev, y_prev) * x_prev + du_dy(x_prev, y_prev) * y_prev
    coef_b = -v(x_prev, y_prev) + dv_dx(x_prev, y_prev) * x_prev + dv_dy(x_prev, y_prev) * y_prev

    # Compute x_new and y_new
    try:
        x_new = (coef_a * dv_dy(x_prev, y_prev) - coef_b * du_dy(x_prev, y_prev)) / jacobian(x_prev, y_prev)
        y_new = (coef_b * du_dx(x_prev, y_prev) - coef_a * dv_dx(x_prev, y_prev)) / jacobian(x_prev, y_prev)
    except ZeroDivisionError:
        print("Error: the Jacobian is zero, cannot continue.")
        break

    # Check whether the solution is exact
    if u(x_new, y_new) == 0 and v(x_new, y_new) == 0:
        converged = 1

    # Compute errors
    error_x = abs((x_new - x_prev) / x_new)
    error_y = abs((y_new - y_prev) / y_new)

    # Check whether the error is below the threshold
    if error_x < error_tol and error_y < error_tol:
        converged = 1

    # Check the Jacobian at the new iterate
    if jacobian(x_new, y_new) == 0:
        valid_bracket = 0

    # Update values
    x_prev = x_new
    y_prev = y_new
    iterations += 1

# Final results
print("\nThe circles intersect approximately at:")
print(f"x = {x_new:.6f}, y = {y_new:.6f}")

print("\nEvaluation at the solution:")
print(f"u(x, y) = {u(x_new, y_new):.6f}")
print(f"v(x, y) = {v(x_new, y_new):.6f}")

print("\nIndividual errors:")
print(f"Error in x: {error_x:.6e}")
print(f"Error in y: {error_y:.6e}")

print("\nIterations performed:")
print(iterations)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Visualization of problem A: the two circles and their intersection point,
# recomputed here from a fixed starting guess (x0=1.0, y0=3.8) so this plot
# can be generated without the interactive input() above.
def u_viz(x, y):
    return (x-4)**2 + (y-4)**2 - 4

def v_viz(x, y):
    return x**2 + y**2 - 16

delta_viz = 1e-4

def du_dx_viz(x, y): return (u_viz(x + delta_viz, y) - u_viz(x, y)) / delta_viz
def du_dy_viz(x, y): return (u_viz(x, y + delta_viz) - u_viz(x, y)) / delta_viz
def dv_dx_viz(x, y): return (v_viz(x + delta_viz, y) - v_viz(x, y)) / delta_viz
def dv_dy_viz(x, y): return (v_viz(x, y + delta_viz) - v_viz(x, y)) / delta_viz
def jacobian_viz(x, y): return du_dx_viz(x, y) * dv_dy_viz(x, y) - du_dy_viz(x, y) * dv_dx_viz(x, y)

x_prev, y_prev = 1.0, 3.8
error_x = error_y = 1
error_tol_viz = 1e-6
iterations_viz = 0
while iterations_viz < 1000 and (error_x > error_tol_viz or error_y > error_tol_viz):
    coef_a = -u_viz(x_prev, y_prev) + du_dx_viz(x_prev, y_prev) * x_prev + du_dy_viz(x_prev, y_prev) * y_prev
    coef_b = -v_viz(x_prev, y_prev) + dv_dx_viz(x_prev, y_prev) * x_prev + dv_dy_viz(x_prev, y_prev) * y_prev
    j = jacobian_viz(x_prev, y_prev)
    x_new = (coef_a * dv_dy_viz(x_prev, y_prev) - coef_b * du_dy_viz(x_prev, y_prev)) / j
    y_new = (coef_b * du_dx_viz(x_prev, y_prev) - coef_a * dv_dx_viz(x_prev, y_prev)) / j
    error_x = abs((x_new - x_prev) / x_new)
    error_y = abs((y_new - y_prev) / y_new)
    x_prev, y_prev = x_new, y_new
    iterations_viz += 1

theta = np.linspace(0, 2 * np.pi, 400)
circle1_x = 4 + 2 * np.cos(theta)
circle1_y = 4 + 2 * np.sin(theta)
circle2_x = 4 * np.cos(theta)
circle2_y = 4 * np.sin(theta)

plt.figure(figsize=(8, 8))
plt.plot(circle1_x, circle1_y, label=r'$(x-4)^2+(y-4)^2=4$', color='blue')
plt.plot(circle2_x, circle2_y, label=r'$x^2+y^2=16$', color='green')
plt.scatter([x_prev], [y_prev], color='red', s=100, zorder=5, label=f'Root: ({x_prev:.4f}, {y_prev:.4f})')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Newton-Raphson: Intersection of Two Circles')
plt.axis('equal')
plt.grid(True)
plt.legend()
plt.savefig("circles_intersection.png", dpi=150, bbox_inches="tight")
plt.show()


### B. Find a root of the following system of equations:
$$
x = y + x^2 - 0.5 = 4
$$

$$
y = x^2 - 5xy
$$

The expression was likewise rearranged into two equations set equal to zero:
$$
y + x^2 - 4.5 = 0
$$

$$
y - x^2 + 5xy = 0
$$

In [ ]:
import numpy as np

# Functions u(x, y) and v(x, y)
def u(x, y):
    return y + x**2 -4.5

def v(x, y):
    return y - x**2 + 5*x*y

# General parameters
delta = 1e-4
error_tol = 1e-6
max_attempts = 2
max_iterations = 1000

# Approximate derivatives
def du_dx(x, y):
    return (u(x + delta, y) - u(x, y)) / delta

def du_dy(x, y):
    return (u(x, y + delta) - u(x, y)) / delta

def dv_dx(x, y):
    return (v(x + delta, y) - v(x, y)) / delta

def dv_dy(x, y):
    return (v(x, y + delta) - v(x, y)) / delta

# Jacobian determinant
def jacobian(x, y):
    return du_dx(x, y) * dv_dy(x, y) - du_dy(x, y) * dv_dx(x, y)

# Flags and errors
valid_bracket = 0
error_x = 1
error_y = 1
converged = 0
attempts = 0

# First loop: propose initial values
while valid_bracket == 0 and converged == 0 and attempts < max_attempts:
    try:
        x0 = float(input("Give me the initial value for x: "))
        y0 = float(input("Give me the initial value for y: "))
        
        if jacobian(x0, y0) != 0:
            valid_bracket = 1
        
        if u(x0, y0) == 0 and v(x0, y0) == 0:
            converged = 1

        attempts += 1

    except ValueError:
        print("Invalid input. Please enter numeric values.")

# Second loop: iterative method
x_prev, y_prev = x0, y0
iterations = 0

while valid_bracket == 1 and converged == 0 and iterations < max_iterations:
    coef_a = -u(x_prev, y_prev) + du_dx(x_prev, y_prev) * x_prev + du_dy(x_prev, y_prev) * y_prev
    coef_b = -v(x_prev, y_prev) + dv_dx(x_prev, y_prev) * x_prev + dv_dy(x_prev, y_prev) * y_prev

    # Compute x_new and y_new
    try:
        x_new = (coef_a * dv_dy(x_prev, y_prev) - coef_b * du_dy(x_prev, y_prev)) / jacobian(x_prev, y_prev)
        y_new = (coef_b * du_dx(x_prev, y_prev) - coef_a * dv_dx(x_prev, y_prev)) / jacobian(x_prev, y_prev)
    except ZeroDivisionError:
        print("Error: the Jacobian is zero, cannot continue.")
        break

    # Check whether the solution is exact
    if u(x_new, y_new) == 0 and v(x_new, y_new) == 0:
        converged = 1

    # Compute errors
    error_x = abs((x_new - x_prev) / x_new)
    error_y = abs((y_new - y_prev) / y_new)

    # Check whether the error is below the threshold
    if error_x < error_tol and error_y < error_tol:
        converged = 1

    # Check the Jacobian at the new iterate
    if jacobian(x_new, y_new) == 0:
        valid_bracket = 0

    # Update values
    x_prev = x_new
    y_prev = y_new
    iterations += 1

# Final results
print("\nApproximate root of the system:")
print(f"x = {x_new:.6f}, y = {y_new:.6f}")

print("\nEvaluation at the solution:")
print(f"u(x, y) = {u(x_new, y_new):.6f}")
print(f"v(x, y) = {v(x_new, y_new):.6f}")

print("\nIndividual errors:")
print(f"Error in x: {error_x:.6e}")
print(f"Error in y: {error_y:.6e}")

print("\nIterations performed:")
print(iterations)


### C. Find a root of the following system of equations:
$$
3x - \cos(y z) - \frac{1}{2} = 0
$$

$$
4x^2 - 625y^2 + 2y - 1 = 0
$$

$$
e^{-xy} + 20z + \frac{10\pi - 3}{3} = 0
$$

In [ ]:

# Functions u(x, y, z), v(x, y, z) and w(x, y, z)
def u(x, y, z):
    return 3 * x - np.cos(y * z) - 1/2

def v(x, y, z):
    return 4 * x**2 - 625 * y**2 + 2 * y - 1

def w(x, y, z):
    return np.exp(-x * y) + 20 * z + (10 * np.pi - 3) / 3

# General parameters
delta = 1e-4
error_tol = 1e-6
max_attempts = 2
max_iterations = 1000

# Approximate derivatives
def du_dx(x, y, z):
    return (u(x + delta, y, z) - u(x, y, z)) / delta

def du_dy(x, y, z):
    return (u(x, y + delta, z) - u(x, y, z)) / delta

def du_dz(x, y, z):
    return (u(x, y, z + delta) - u(x, y, z)) / delta

def dv_dx(x, y, z):
    return (v(x + delta, y, z) - v(x, y, z)) / delta

def dv_dy(x, y, z):
    return (v(x, y + delta, z) - v(x, y, z)) / delta

def dv_dz(x, y, z):
    return (v(x, y, z + delta) - v(x, y, z)) / delta

def dw_dx(x, y, z):
    return (w(x + delta, y, z) - w(x, y, z)) / delta

def dw_dy(x, y, z):
    return (w(x, y + delta, z) - w(x, y, z)) / delta

def dw_dz(x, y, z):
    return (w(x, y, z + delta) - w(x, y, z)) / delta

# Jacobian determinant
def jacobian(x, y, z):
    return (
        du_dx(x, y, z) * (dv_dy(x, y, z) * dw_dz(x, y, z) - dv_dz(x, y, z) * dw_dy(x, y, z)) +
        du_dy(x, y, z) * (dv_dz(x, y, z) * dw_dx(x, y, z) - dv_dx(x, y, z) * dw_dz(x, y, z)) +
        du_dz(x, y, z) * (dv_dx(x, y, z) * dw_dy(x, y, z) - dv_dy(x, y, z) * dw_dx(x, y, z))
    )

# Flags and errors
valid_bracket = 0
error_x = 1
error_y = 1
error_z = 1
converged = 0
attempts = 0

# First loop: propose initial values
while valid_bracket == 0 and converged == 0 and attempts < max_attempts:
    try:
        x0 = float(input("Give me the initial value for x: "))
        y0 = float(input("Give me the initial value for y: "))
        z0 = float(input("Give me the initial value for z: "))

        if jacobian(x0, y0, z0) != 0:
            valid_bracket = 1
        
        if u(x0, y0, z0) == 0 and v(x0, y0, z0) == 0 and w(x0, y0, z0) == 0:
            converged = 1

        attempts += 1

    except ValueError:
        print("Invalid input. Please enter numeric values.")

# Second loop: iterative method
x_prev, y_prev, z_prev = x0, y0, z0
iterations = 0

while valid_bracket == 1 and converged == 0 and iterations < max_iterations:
    # Coefficients
    coef_a = -u(x_prev, y_prev, z_prev) + du_dx(x_prev, y_prev, z_prev) * x_prev + du_dy(x_prev, y_prev, z_prev) * y_prev + du_dz(x_prev, y_prev, z_prev) * z_prev
    coef_b = -v(x_prev, y_prev, z_prev) + dv_dx(x_prev, y_prev, z_prev) * x_prev + dv_dy(x_prev, y_prev, z_prev) * y_prev + dv_dz(x_prev, y_prev, z_prev) * z_prev
    coef_c = -w(x_prev, y_prev, z_prev) + dw_dx(x_prev, y_prev, z_prev) * x_prev + dw_dy(x_prev, y_prev, z_prev) * y_prev + dw_dz(x_prev, y_prev, z_prev) * z_prev

    # Compute x_new, y_new and z_new
    try:
        x_new = (coef_a * (dv_dy(x_prev, y_prev, z_prev) * dw_dz(x_prev, y_prev, z_prev) - dv_dz(x_prev, y_prev, z_prev) * dw_dy(x_prev, y_prev, z_prev)) -
                   coef_b * (du_dy(x_prev, y_prev, z_prev) * dw_dz(x_prev, y_prev, z_prev) - du_dz(x_prev, y_prev, z_prev) * dw_dy(x_prev, y_prev, z_prev)) +
                   coef_c * (du_dy(x_prev, y_prev, z_prev) * dv_dz(x_prev, y_prev, z_prev) - du_dz(x_prev, y_prev, z_prev) * dv_dy(x_prev, y_prev, z_prev))) / jacobian(x_prev, y_prev, z_prev)

        y_new = (coef_b * (du_dx(x_prev, y_prev, z_prev) * dw_dz(x_prev, y_prev, z_prev) - du_dz(x_prev, y_prev, z_prev) * dw_dx(x_prev, y_prev, z_prev)) -
                   coef_a * (dv_dx(x_prev, y_prev, z_prev) * dw_dz(x_prev, y_prev, z_prev) - dv_dz(x_prev, y_prev, z_prev) * dw_dx(x_prev, y_prev, z_prev)) +
                   coef_c * (dv_dx(x_prev, y_prev, z_prev) * du_dz(x_prev, y_prev, z_prev) - du_dx(x_prev, y_prev, z_prev) * dv_dz(x_prev, y_prev, z_prev))) / jacobian(x_prev, y_prev, z_prev)

        z_new = (coef_c * (du_dx(x_prev, y_prev, z_prev) * dv_dy(x_prev, y_prev, z_prev) - du_dy(x_prev, y_prev, z_prev) * dv_dx(x_prev, y_prev, z_prev)) -
                   coef_a * (dv_dx(x_prev, y_prev, z_prev) * dw_dy(x_prev, y_prev, z_prev) - dv_dy(x_prev, y_prev, z_prev) * dw_dx(x_prev, y_prev, z_prev)) +
                   coef_b * (du_dx(x_prev, y_prev, z_prev) * dw_dy(x_prev, y_prev, z_prev) - du_dy(x_prev, y_prev, z_prev) * dw_dx(x_prev, y_prev, z_prev))) / jacobian(x_prev, y_prev, z_prev)
    except ZeroDivisionError:
        print("Error: the Jacobian is zero, cannot continue.")
        break

    # Check whether the solution is exact
    if u(x_new, y_new, z_new) == 0 and v(x_new, y_new, z_new) == 0 and w(x_new, y_new, z_new) == 0:
        converged = 1

    # Compute errors
    error_x = abs((x_new - x_prev) / x_new)
    error_y = abs((y_new - y_prev) / y_new)
    error_z = abs((z_new - z_prev) / z_new)

    # Check whether the error is below the threshold
    if error_x < error_tol and error_y < error_tol and error_z < error_tol:
        converged = 1

    # Check the Jacobian at the new iterate
    if jacobian(x_new, y_new, z_new) == 0:
        valid_bracket = 0

    # Update values
    x_prev = x_new
    y_prev = y_new
    z_prev = z_new
    iterations += 1

# Final results
print("\nSolution:")
print(f"x = {x_new:.6f}, y = {y_new:.6f}, z = {z_new:.6f}")

print("\nEvaluation at the solution:")
print(f"u(x, y, z) = {u(x_new, y_new, z_new):.6f}")
print(f"v(x, y, z) = {v(x_new, y_new, z_new):.6f}")
print(f"w(x, y, z) = {w(x_new, y_new, z_new):.6f}")

print("\nIndividual errors:")
print(f"Error in x: {error_x:.6e}")
print(f"Error in y: {error_y:.6e}")
print(f"Error in z: {error_z:.6e}")

print("\nIterations performed:")
print(iterations)
